## Environment Setup

To kick off the project, I began by verifying that my Spark session was active and running the correct runtime. This project is executed entirely on Databricks Community Edition, with Spark version 3.3.2.
Using the built-in `spark` object and `.version` method, I ensured the cluster was up and ready for distributed ETL operations

In [0]:
# Confirm Spark session is active
spark

SparkSession - hive 
 
 
 SparkContext 

 Spark UI 

 
 Version 
 v3.3.2 
 Master 
 local[8] 
 AppName 
 Databricks Shell

In [0]:
# Show Spark Version
spark.version

Out[2]: '3.3.2'

## Uploading Data to Databrick File System (DBFS)

I uploaded the raw dataset('hotel_bookings.csv') using the 'Create Table' workflow in the Catalog tab. This stored the file in the Databrick File System at:

`/FileStore/tables/hotel_bookings.csv`

This method makes it easy to reference the file when loading it into a PySpark DataFrame for transformation and analysis.

## Loading Data into a Spark DataFrame

I loaded the dataset using PySpark's `read.format('csv')` API, applying key options like:
* `header = True` : to treat the first row as column names
* `inferSchema = False`: initially turned off for full control
* `sep = ","` : to match CSV formatting

This approach gives me flexibility for schema enforcement and allows optimized reading for large scale ETL pipelines.

In [0]:
# Load the data
df = spark.read.csv('/FileStore/tables/hotel_bookings.csv', header=True, inferSchema=False, sep = ',')
df.display()


hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
Resort Hotel,0,342,2015,July,27,1,0,0,2,0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,NULL,NULL,0,Transient,0,0,0,Check-Out,2015-07-01
Resort Hotel,0,737,2015,July,27,1,0,0,2,0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,NULL,NULL,0,Transient,0,0,0,Check-Out,2015-07-01
Resort Hotel,0,7,2015,July,27,1,0,1,1,0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,NULL,NULL,0,Transient,75,0,0,Check-Out,2015-07-02
Resort Hotel,0,13,2015,July,27,1,0,1,1,0,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,304,NULL,0,Transient,75,0,0,Check-Out,2015-07-02
Resort Hotel,0,14,2015,July,27,1,0,2,2,0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240,NULL,0,Transient,98,0,1,Check-Out,2015-07-03
Resort Hotel,0,14,2015,July,27,1,0,2,2,0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240,NULL,0,Transient,98,0,1,Check-Out,2015-07-03
Resort Hotel,0,0,2015,July,27,1,0,2,2,0,0,BB,PRT,Direct,Direct,0,0,0,C,C,0,No Deposit,NULL,NULL,0,Transient,107,0,0,Check-Out,2015-07-03
Resort Hotel,0,9,2015,July,27,1,0,2,2,0,0,FB,PRT,Direct,Direct,0,0,0,C,C,0,No Deposit,303,NULL,0,Transient,103,0,1,Check-Out,2015-07-03
Resort Hotel,1,85,2015,July,27,1,0,3,2,0,0,BB,PRT,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240,NULL,0,Transient,82,0,1,Canceled,2015-05-06
Resort Hotel,1,75,2015,July,27,1,0,3,2,0,0,HB,PRT,Offline TA/TO,TA/TO,0,0,0,D,D,0,No Deposit,15,NULL,0,Transient,105.5,0,0,Canceled,2015-04-22


##  Reloading the Dataset After Cluster Restart

Since Databricks clusters are ephemeral, I reloaded the original CSV file from the Databricks File System (`/FileStore/tables/hotel_bookings.csv`) using Spark’s `.read()` API.

I re-enabled schema inference and header detection to restore the original DataFrame structure.


In [0]:
# Reload the dataset into Spark
#df = spark.read.option("header", True).option("inferSchema", True).csv("/FileStore/tables/hotel_bookings.csv")


# Registering Temp View for SQL Queries

To leverage SQL-style queries within Spark, I created a temporary view of the DataFrame using `createOrReplaceTempView()`

This makes it easy to run SELECT statements and quickly inspect the dataset in a familiar SQL format, which is helpful when cross-verifying transformations.

In [0]:
# Create a view or table

temp_table_name = "hotel_bookings_raw"

df.createOrReplaceTempView(temp_table_name)

In [0]:
%sql

/* Query the created temp table in a SQL cell */

select * from `hotel_bookings_raw` LIMIT 10

hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
Resort Hotel,0,342,2015,July,27,1,0,0,2,0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,NULL,NULL,0,Transient,0,0,0,Check-Out,2015-07-01
Resort Hotel,0,737,2015,July,27,1,0,0,2,0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,NULL,NULL,0,Transient,0,0,0,Check-Out,2015-07-01
Resort Hotel,0,7,2015,July,27,1,0,1,1,0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,NULL,NULL,0,Transient,75,0,0,Check-Out,2015-07-02
Resort Hotel,0,13,2015,July,27,1,0,1,1,0,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,304,NULL,0,Transient,75,0,0,Check-Out,2015-07-02
Resort Hotel,0,14,2015,July,27,1,0,2,2,0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240,NULL,0,Transient,98,0,1,Check-Out,2015-07-03
Resort Hotel,0,14,2015,July,27,1,0,2,2,0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240,NULL,0,Transient,98,0,1,Check-Out,2015-07-03
Resort Hotel,0,0,2015,July,27,1,0,2,2,0,0,BB,PRT,Direct,Direct,0,0,0,C,C,0,No Deposit,NULL,NULL,0,Transient,107,0,0,Check-Out,2015-07-03
Resort Hotel,0,9,2015,July,27,1,0,2,2,0,0,FB,PRT,Direct,Direct,0,0,0,C,C,0,No Deposit,303,NULL,0,Transient,103,0,1,Check-Out,2015-07-03
Resort Hotel,1,85,2015,July,27,1,0,3,2,0,0,BB,PRT,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240,NULL,0,Transient,82,0,1,Canceled,2015-05-06
Resort Hotel,1,75,2015,July,27,1,0,3,2,0,0,HB,PRT,Offline TA/TO,TA/TO,0,0,0,D,D,0,No Deposit,15,NULL,0,Transient,105.5,0,0,Canceled,2015-04-22


In [0]:
# With this registered as a temp view, it will only be available to this particular notebook. If you'd like other users to be able to query this table, you can also create a table from the DataFrame.
# Once saved, this table will persist across cluster restarts as well as allow various users across different notebooks to query this data.
# To do so, choose your table name and uncomment the bottom line.

permanent_table_name = "hotel_bookings_csv"

# df.write.format("parquet").saveAsTable(permanent_table_name)

## Exploring the Schema

Before transforming the data, i inspected the schema using Spark's `printSchema()` method. This helped identify:

* Correct data types for numeric, categorical and date fields.
* Fields like `children`, `agent` and `company` that came in as strings but should be integers
* Potential nullability issue to handle later during cleaning

This step sets the foundation for safe downstream processing and modeling. 

In [0]:
# Inspect the schema

df.printSchema()

root
 |-- hotel: string (nullable = true)
 |-- is_canceled: string (nullable = true)
 |-- lead_time: string (nullable = true)
 |-- arrival_date_year: string (nullable = true)
 |-- arrival_date_month: string (nullable = true)
 |-- arrival_date_week_number: string (nullable = true)
 |-- arrival_date_day_of_month: string (nullable = true)
 |-- stays_in_weekend_nights: string (nullable = true)
 |-- stays_in_week_nights: string (nullable = true)
 |-- adults: string (nullable = true)
 |-- children: string (nullable = true)
 |-- babies: string (nullable = true)
 |-- meal: string (nullable = true)
 |-- country: string (nullable = true)
 |-- market_segment: string (nullable = true)
 |-- distribution_channel: string (nullable = true)
 |-- is_repeated_guest: string (nullable = true)
 |-- previous_cancellations: string (nullable = true)
 |-- previous_bookings_not_canceled: string (nullable = true)
 |-- reserved_room_type: string (nullable = true)
 |-- assigned_room_type: string (nullable = true)
 

## Type Conversion and Null Handling

During my initial schema inspection, I noticed that some numeric fields such as `children`, `agent`, and `company` were loaded as strings. These need to be cast as integers for modeling and aggregation.

I also wanted to pre-emptively handle nulls in these fields. Since missing values in `agent` and `company` typically mean "not specified", I chose to fill them with zero.

For `children`, I made the same choice under the assumption that a missing entry likely meant zero children were present.



In [0]:
from pyspark.sql.functions import col, when

# Convert `childer`, `agent` and `company` to numeric safely
df = df.withColumn('children', col('children').cast('integer')) \
    .withColumn('agent', col('agent').cast('integer')) \
        .withColumn('company', col('company').cast('integer'))

# Fill missing values with zeros (assumption: missing means unknown = 0)
df = df.fillna({
    'children' : 0,
    'agent' : 0,
    'company' : 0
    })        

## Month Name to Number Conversion

To enable easier sorting and time-based analysis, I needed to convert the `arrival_date_month` column from month names (e.g., "January", "February") to numeric values (e.g., 1, 2, ..., 12).

Instead of using chained `.when()` conditions, I defined a dictionary mapping each month to its corresponding number and used `create_map()` from `pyspark.sql.functions` to apply the transformation in a cleaner and more scalable way.


In [0]:
from pyspark.sql.functions import create_map, lit

# Define mapping of month names to their numerical values
month_map = {
    "January": 1, "February": 2, "March": 3, "April": 4,
    "May": 5, "June": 6, "July": 7, "August": 8,
    "September": 9, "October": 10, "November": 11, "December": 12
}

# Convert the dictionary into a Spark Map expression
month_expr = create_map([lit(x) for x in sum(month_map.items(), ())])

# Create a new column with numeric month values
df = df.withColumn("arrival_month_num", month_expr[col("arrival_date_month")])

# Quick check
df.select("arrival_date_month", "arrival_month_num").show(12)


+------------------+-----------------+
|arrival_date_month|arrival_month_num|
+------------------+-----------------+
|              July|                7|
|              July|                7|
|              July|                7|
|              July|                7|
|              July|                7|
|              July|                7|
|              July|                7|
|              July|                7|
|              July|                7|
|              July|                7|
|              July|                7|
|              July|                7|
+------------------+-----------------+
only showing top 12 rows



## Missing Values Overview

Before proceeding with EDA or modeling, I wanted to ensure data quality by identifying and treating missing values. Using Spark'a `select().agg()` function, I performed a null count check across all columns. This gave me a clear view of where data imputation or cleaning was required.

In [0]:
from pyspark.sql.functions import sum as spark_sum

# Create a dictionary of null counts for each column
null_counts = df.select([spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df.columns])
null_counts.show(vertical = True)


-RECORD 0-----------------------------
 hotel                          | 0   
 is_canceled                    | 0   
 lead_time                      | 0   
 arrival_date_year              | 0   
 arrival_date_month             | 0   
 arrival_date_week_number       | 0   
 arrival_date_day_of_month      | 0   
 stays_in_weekend_nights        | 0   
 stays_in_week_nights           | 0   
 adults                         | 0   
 children                       | 0   
 babies                         | 0   
 meal                           | 0   
 country                        | 0   
 market_segment                 | 0   
 distribution_channel           | 0   
 is_repeated_guest              | 0   
 previous_cancellations         | 0   
 previous_bookings_not_canceled | 0   
 reserved_room_type             | 0   
 assigned_room_type             | 0   
 booking_changes                | 0   
 deposit_type                   | 0   
 agent                          | 0   
 company                 

### Null Handling Summary

Interestingly after conversion of certain string fields like `children`, `agent` and `company` to integers, and filling their missing values with zero, the dataset now contains **no nulls**.

This confirmed that earlier data-cleaning steps were effective. As a result, I can confidently proceed to the next phase without any further imputation. 

No columns were dropped or forward/backward filled - the null-handling strategy was targeted, conservative, and based on context-aware assumptions.

---

## Duplicate Row Check

To ensure the integrity of the dataset, I checked for exact duplicate rows. While some booking details might look similar, complete row duplication could indicate pipeline redundancy or accidental re-ingestion.

Since PySpark doesn’t offer a `.duplicated()` method like pandas, I used `groupBy` on all columns and counted how many times each row appeared. Any row with a count greater than 1 was flagged as a duplicate.


In [0]:
# Find duplicate rows by grouping all columns and counting
duplicate_rows = df.groupBy(df.columns).count().filter("count > 1")

# Show any duplicates
duplicate_rows.show(truncate=False)

# Count how many duplicates exist
duplicate_count = duplicate_rows.count()
print(f"Number of duplicate rows: {duplicate_count}")


+------------+-----------+---------+-----------------+------------------+------------------------+-------------------------+-----------------------+--------------------+------+--------+------+----+-------+--------------+--------------------+-----------------+----------------------+------------------------------+------------------+------------------+---------------+------------+-----+-------+--------------------+---------------+------+---------------------------+-------------------------+------------------+-----------------------+-----------------+-----+
|hotel       |is_canceled|lead_time|arrival_date_year|arrival_date_month|arrival_date_week_number|arrival_date_day_of_month|stays_in_weekend_nights|stays_in_week_nights|adults|children|babies|meal|country|market_segment|distribution_channel|is_repeated_guest|previous_cancellations|previous_bookings_not_canceled|reserved_room_type|assigned_room_type|booking_changes|deposit_type|agent|company|days_in_waiting_list|customer_type  |adr   |re

## Removing Duplicate Rows

After checking for full-row duplicates, I found that 8,171 rows were duplicated across all columns. This likely occurred due to upstream data merging or re-ingestion.

To preserve data integrity, I removed these duplicates using Spark’s `dropDuplicates()` function, which is optimized for distributed datasets.

In [0]:
# Drop exact duplicates
df = df.dropDuplicates()

# Confirm row counts after cleaning
print(f'Row counts after dropping duplicates: {df.count()}')

Row counts after dropping duplicates: 87396


## Target Variable Distribution: `is_canceled`

To begin the EDA phase, I analyzed the distribution of the target variable — whether a hotel booking was canceled (`1`) or honored (`0`).

Understanding the class balance is important to inform model selection, evaluation metric prioritization, and whether class imbalance techniques may be needed.


In [0]:
# Count how many bookings were canceled vs not canceled
df.groupBy("is_canceled").count().orderBy("is_canceled").show()

+-----------+-----+
|is_canceled|count|
+-----------+-----+
|          0|63371|
|          1|24025|
+-----------+-----+



### **Observation:**  
Roughly **27.5% of the bookings were canceled**, while **72.5% were fulfilled**. This is moderately imbalanced and may affect baseline model behavior and metric interpretation later on.

During modeling, I plan to track **recall and F1-score** closely to ensure cancellation cases are properly detected.

---


## Exporting Cleaned Dataset for Local Modeling

After completing all core data cleaning steps in PySpark(type conversion, null handling, duplicate removal, and date column creation), I exported the final dataset to a CSV file.

This allows me to switch ove to my local environment (VSCode), for exploratory analysis, modeling, and visualization using Python libraries like pandas, scikit-learn, and seaborn.

In [0]:
# Save cleaned dataset to DBFS as CSV
df.coalesce(1) \
  .write.mode("overwrite") \
  .option("header", True) \
  .csv("/FileStore/hotel_cleaned_singlefile")


In [0]:
# import os

# dbutils.fs.ls("/FileStore/hotel_cleaned/")